# 🎭 Ролевая модель на базе Qwen (Qwen3.5-2B и Qwen2.5-1.5B)

**Модель уже дообучена** — обучать ничего не нужно.

1. Включи видеокарту: меню «Среда выполнения» → «Сменить среду выполнения» → **T4 GPU** → Сохранить.
2. Нажми **▶ у ячейки «💬 Чат»** — через 2–3 минуты появится ссылка `https://….gradio.live`. Открой её — это чат во весь экран.
3. Персонажа задавай в «🎭 Персонаж и настройки» (под окном чата), действия пиши в звёздочках: `*улыбается*`.

⚠️ Бесплатный Colab **выключается, если долго ничего не делать** — тогда ссылка перестаёт работать. Вернись сюда, нажми ▶ ещё раз и открой **новую** ссылку.

Модель «base» — без встроенных отказов. Защита от сексуальных сцен с несовершеннолетними включена всегда.

In [ ]:
#@title 💬 Чат с ролевой Qwen (как в ChatGPT)
модель = "Qwen3.5-2B, новая (из GitHub)" #@param ["Qwen3.5-2B, новая (из GitHub)", "Qwen2.5-1.5B, прошлая (из GitHub)", "своё обучение (с Google Диска)", "Qwen3.5-2B без адаптера (чистая)"]
import os, sys, torch
REPO, BRANCH = '/content/ai', 'ccr-d2dd0252-6ksj1l'
SAVE = '/content/drive/MyDrive/ai_qwen'      # сюда сохраняет ячейка «Дообучить»
if not torch.cuda.is_available():
    print('⚠️ Видеокарта не включена — ответы будут ОЧЕНЬ медленными (около минуты).\n'
          '   Лучше: «Среда выполнения» → «Сменить среду выполнения» → T4 GPU → Сохранить, потом снова ▶')
# 1) самая свежая версия проекта (вместе с готовым адаптером)
if os.path.exists(REPO):
    !git -C {REPO} fetch -q origin {BRANCH} && git -C {REPO} reset -q --hard FETCH_HEAD
else:
    !git clone -q -b {BRANCH} https://github.com/r0561952-beep/ai.git {REPO}
os.chdir(REPO)
# 2) библиотеки
!pip install -q "transformers==5.24.1" "peft==0.22.0" "accelerate==1.16.0" "gradio>=6,<7"
if REPO not in sys.path:
    sys.path.insert(0, REPO)
for m in ['model', 'tokenizer_utils', 'chat', 'web_chat', 'qwen_chat', 'rp_card', 'safety']:
    sys.modules.pop(m, None)  # забываем старые версии кода
# 3) какой адаптер подключить (базовая модель записана в нём самом)
adapter = f'{REPO}/model/qwen35_rp_adapter'
if not os.path.exists(adapter) or модель.startswith('Qwen2.5'):
    adapter = f'{REPO}/model/qwen_rp_adapter'
if модель.startswith('своё'):
    from google.colab import drive
    drive.mount('/content/drive')
    if os.path.exists(f'{SAVE}/adapter'):
        adapter = f'{SAVE}/adapter'
    else:
        print('⚠️ На Диске своего адаптера нет — беру готовый из GitHub.')
elif 'без адаптера' in модель:
    adapter = None
if adapter and not os.path.exists(adapter):
    print('⚠️ Адаптера нет в репозитории — запускаю чистую Qwen.')
    adapter = None
# 4) загружаем модель (первый раз скачивается 3–4,5 ГБ) и открываем чат
from qwen_chat import QwenBot
from web_chat import launch
bot = QwenBot(adapter, base=None if adapter else 'Qwen/Qwen3.5-2B-Base')
print('Модель:', bot.label)
print('👇 Открой ссылку «public URL» ниже')
launch(share=True, bot=bot)

---
### Дополнительно (не обязательно)
Своё обучение: результат сохраняется на Google Диск (`ai_qwen`), в чате выбери «своё обучение».

In [ ]:
#@title 🔧 Дообучить заново самому (НЕ обязательно, ~2 часа на T4; после отключения — просто ▶ снова)
import os, sys, torch
SAVE = '/content/drive/MyDrive/ai_qwen'      # здесь хранится прогресс и готовый адаптер
REPO, BRANCH = '/content/ai', 'ccr-d2dd0252-6ksj1l'
if not torch.cuda.is_available():
    print('⚠️ Нет видеокарты! Меню «Среда выполнения» → «Сменить среду выполнения» → T4 GPU → Сохранить. Потом снова ▶')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(SAVE, exist_ok=True)
    if os.path.exists(REPO):
        !git -C {REPO} fetch -q origin {BRANCH} && git -C {REPO} reset -q --hard FETCH_HEAD
    else:
        !git clone -q -b {BRANCH} https://github.com/r0561952-beep/ai.git {REPO}
    os.chdir(REPO)
    !pip install -q "transformers==5.24.1" "peft==0.22.0" "accelerate==1.16.0" zstandard
    # ролевые данные готовим один раз и храним на Диске
    os.makedirs(f'{SAVE}/data', exist_ok=True)
    if not os.path.exists(f'{SAVE}/data/rp18_ru.jsonl.gz'):
        print('Первый запуск: скачиваю и фильтрую ролевые данные (~5 минут)...')
        !python prepare_rp.py
        !cp data/rp_ru.jsonl.gz data/rp18_ru.jsonl.gz {SAVE}/data/
    else:
        !cp {SAVE}/data/rp_ru.jsonl.gz {SAVE}/data/rp18_ru.jsonl.gz data/
    !python qwen_train.py --base Qwen/Qwen2.5-1.5B --out {SAVE} 2>&1 | tee -a {SAVE}/train.log

In [ ]:
#@title 📈 Журнал обучения
import os
SAVE = '/content/drive/MyDrive/ai_qwen'
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
log = f'{SAVE}/train.log'
if os.path.exists(log):
    lines = [l for l in open(log, encoding='utf-8') if l.startswith(('шаг', 'продолжаю', '✅', 'лимит', 'примеров'))]
    print(''.join(lines[-12:]))
else:
    print('Журнала пока нет — сначала запусти обучение.')